# 🚗 Sistem Deteksi Pelat Nomor, Klasifikasi Warna Kendaraan & Status Pajak (ALPR)
### Notebook Eksekusi & Evaluasi Bertahap (`run.ipynb` — Mendukung VS Code Google Colab GPU & Lokal)

Notebook ini dirancang agar dapat dijalankan langsung di **VS Code menggunakan ekstensi Google Colab (GPU T4/A100)** maupun di kernel Python lokal. Setiap tahapan menampilkan **visualisasi citra hasil** sekaligus **metrik Box Evaluasi** sesuai diagram alur sistem yang baru:
1. **Persiapan Environment & Akses Dataset via Kaggle API (`kagglehub`)** *(Fallback otomatis ke folder lokal jika offline)*
2. **Blok YOLOv8n (Tahap 1–3):** Deteksi & *Crop* ROI $\rightarrow$ Koreksi Kemiringan (*Deskewing*) $\rightarrow$ **[Kebaruan 1] Identifikasi Warna Pelat (HSV) & Klasifikasi Jenis Kendaraan** + Standarisasi Polaritas (Background Hitam & Tulisan Putih)
3. **Blok Peningkatan Kualitas Citra (Tahap 4–6):** *Grayscale & Bilateral Smoothing* $\rightarrow$ *Conditional CLAHE* $\rightarrow$ *Penajaman Citra (Unsharp Masking)*
4. **Blok Segmentasi & Morfologi (Tahap 7–9):** *Segmentasi Biner* $\rightarrow$ *Perbaikan Morfologi* $\rightarrow$ **[Kebaruan 2 Bagian A] Pemisahan Dual-ROI (Nomor Utama vs Angka Pajak `MM.YY`)**
5. **Blok Pengenalan Teks OCR (Tahap 10):** Komparasi 3 Kandidat Input OCR, Validasi Regex & **[Kebaruan 2 Bagian B] Pembacaan Masa Pajak (`MM.YY`) + Klasifikasi Pajak Aktif vs Mati**
6. **Visualisasi Output Akhir & Ringkasan Box Evaluasi 10 Tahap (Tahap 11)**

---
## 0. Instalasi Paket & Setup Modul (Khusus Runtime Colab di VS Code)
Sel ini memeriksa apakah pustaka (`ultralytics`, `easyocr`, `kagglehub`) sudah terpasang di runtime Colab/lokal.

In [ ]:
import os
import sys
import subprocess

def ensure_packages(pkgs):
    for mod_name, pip_name in pkgs:
        try:
            __import__(mod_name)
        except ImportError:
            print(f"[Setup] Menginstal {pip_name}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_packages([
    ("kagglehub", "kagglehub"),
    ("ultralytics", "ultralytics"),
    ("easyocr", "easyocr"),
    ("cv2", "opencv-python-headless"),
    ("matplotlib", "matplotlib"),
])

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Environment siap! Direktori kerja: {project_root}")

---
## 1. Akses Dataset via Kaggle API (`kagglehub`) dengan Fallback ke Lokal
Sesuai arsitektur yang disarankan, sistem **mengutamakan akses langsung melalui API `kagglehub`** (sehingga saat dijalankan menggunakan GPU Colab di VS Code, dataset langsung ditarik ke penyimpanan cepat Colab tanpa perlu mengunggah folder `Dataset/` dari laptop). Jika akses API gagal/offline, sistem otomatis beralih ke folder `Dataset/` lokal.

In [ ]:
import glob
import kagglehub

dataset_path = None

# PRIORITAS 1: Akses langsung lewat Kaggle API (kagglehub)
try:
    print("🌐 Mengakses dataset melalui Kaggle API (kagglehub)...")
    path = kagglehub.dataset_download("juanthomaswijaya/indonesian-license-plate-dataset")
    print("Path to dataset files:", path)
    dataset_path = path
except Exception as e:
    print(f"⚠️ Akses Kaggle API tidak tersedia ({e}). Beralih ke direktori Dataset lokal...")

# PRIORITAS 2 (FALLBACK): Gunakan folder Dataset di lokal jika API gagal
if not dataset_path or not os.path.exists(dataset_path):
    local_candidates = [
        os.path.join(project_root, "Dataset", "images"),
        os.path.join(project_root, "Dataset", "Raw"),
        os.path.join(project_root, "Dataset"),
    ]
    for cand in local_candidates:
        if os.path.exists(cand):
            dataset_path = cand
            print(f"📁 Menggunakan dataset lokal di: {dataset_path}")
            break

# Kumpulkan daftar gambar uji (test set)
test_images = []
if dataset_path:
    test_images = sorted(glob.glob(os.path.join(dataset_path, "**", "images", "test", "*.jpg"), recursive=True))
    test_images = [f for f in test_images if "Recognition" not in f]

print(f"📊 Total citra kendaraan pada data uji (test set): {len(test_images)} gambar")

---
## 2. Pemilihan Sampel Citra Uji
Anda dapat memilih indeks gambar dari `test_images` (atau menentukan path gambar sendiri pada variabel `custom_image_path`).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Ganti indeks (misal 0, 1, 5, 10) atau isi custom_image_path dengan path gambar Anda
SAMPLE_INDEX = 0
custom_image_path = None

if custom_image_path and os.path.exists(custom_image_path):
    sample_path = custom_image_path
elif test_images:
    sample_path = test_images[min(SAMPLE_INDEX, len(test_images) - 1)]
else:
    # Buat citra sintetis pelat putih 'B 1995 JVK' dengan pajak '01.28' jika dataset belum tersedia
    sample_path = os.path.join(project_root, "sample_plate_demo.jpg")
    canvas = np.full((220, 520, 3), 235, dtype=np.uint8)
    cv2.rectangle(canvas, (10, 10), (510, 210), (20, 20, 20), 6)
    cv2.putText(canvas, "B 1995 JVK", (45, 125), cv2.FONT_HERSHEY_SIMPLEX, 2.3, (15, 15, 15), 7, cv2.LINE_AA)
    cv2.putText(canvas, "01.28", (205, 185), cv2.FONT_HERSHEY_SIMPLEX, 1.05, (20, 20, 20), 3, cv2.LINE_AA)
    cv2.imwrite(sample_path, canvas)

image_bgr = cv2.imread(sample_path)
image_name = os.path.basename(sample_path)

plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.title(f"Citra Input: {image_name} ({image_bgr.shape[1]}x{image_bgr.shape[0]} px)")
plt.axis("off")
plt.show()

---
## 3. Blok YOLOv8n (Tahap 1 s/d 3): Deteksi, Deskewing, Klasifikasi Warna HSV & Standar Polaritas
Pada tahap ini modul `PlateDetector` (`Script/detector.py`) menjalankan:
1. **Tahap 1 — Deteksi & Pemotongan (*Crop* ROI):** Lokalisasi *bounding box* pelat nomor dan evaluasi rasio aspek ($W/H$).
2. **Tahap 2 — Koreksi Kemiringan (*Deskewing*):** Estimasi sudut kemiringan pelat (*HoughLinesP / minAreaRect*) dan pelurusan *affine* ($\pm 25^\circ$).
3. **Tahap 3 — [Kebaruan 1] Identifikasi Warna Pelat (HSV) & Standar Polaritas:** Mengklasifikasikan warna pelat (**Putih/Hitam = Pribadi, Kuning = Angkutan, Merah = Dinas**) lalu menyeragamkan polaritas menjadi **Background Hitam & Tulisan Putih**.

In [ ]:
from Script.detector import PlateDetector

detector = PlateDetector(conf_threshold=0.35)
detections = detector.detect(image_bgr)

# Fallback jika gambar input merupakan close-up pelat nomor langsung
if not detections:
    ih, iw = image_bgr.shape[:2]
    deskewed, angle, d_stat = detector.deskew_plate(image_bgr)
    c_info = detector.classify_plate_color(deskewed)
    pol_std, is_inv, p_stat = detector.standardize_polarity(deskewed, c_info["plate_color"])
    detections = [{
        "box": (0, 0, iw, ih),
        "confidence": 1.0,
        "class_id": 0,
        "crop": image_bgr.copy(),
        "deskewed": deskewed,
        "skew_angle": angle,
        "deskew_status": d_stat,
        "plate_color": c_info["plate_color"],
        "vehicle_type": c_info["vehicle_type"],
        "hsv_summary": c_info["hsv_summary"],
        "polarity_std": pol_std,
        "is_inverted": is_inv,
        "polarity_status": p_stat,
        "aspect_ratio": round(float(iw) / max(1.0, float(ih)), 2),
        "is_complete": True,
        "status": "Close-up Pelat Lengkap"
    }]

det = detections[0]

# Visualisasi Tahap 1, 2, dan 3 beserta metrik Box Evaluasi
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(cv2.cvtColor(det["crop"], cv2.COLOR_BGR2RGB))
axes[0].set_title(
    f"1. Deteksi & Crop (YOLOv8n)\n"
    f"Conf: {det['confidence']:.2f} | Rasio W/H: {det['aspect_ratio']} ({det['status']})",
    fontsize=10
)
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(det["deskewed"], cv2.COLOR_BGR2RGB))
axes[1].set_title(
    f"2. Koreksi Kemiringan (Deskewing)\n"
    f"Sudut: {det['skew_angle']}° | {det['deskew_status']}",
    fontsize=10
)
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(det["polarity_std"], cv2.COLOR_BGR2RGB))
axes[2].set_title(
    f"3. Standar Polaritas & Warna HSV\n"
    f"Warna: {det['plate_color']} ({det['hsv_summary']})\n{det['vehicle_type']}",
    fontsize=10
)
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("=== EVALUASI BLOK 1 (TAHAP 1 - 3) ===")
print(f"• Koordinat Box YOLO : {det['box']} (Confidence: {det['confidence']:.2f})")
print(f"• Rasio Aspek (W/H)  : {det['aspect_ratio']} -> Status: {det['status']}")
print(f"• Koreksi Deskewing  : {det['skew_angle']}° ({det['deskew_status']})")
print(f"• [KEBARUAN 1] Warna : Pelat {det['plate_color']} ({det['hsv_summary']}) -> {det['vehicle_type']}")
print(f"• Standar Polaritas  : {det['polarity_status']}")

---
## 4. Blok Peningkatan Kualitas Citra (Tahap 4 s/d 6): Grayscale, Smoothing, Conditional CLAHE & Unsharp Masking
Modul `PlatePreprocessor` (`Script/preprocessor.py`) memproses citra `polarity_std` melalui:
4. **Tahap 4 — Grayscale & Smoothing:** Konversi ke *Grayscale*, *upscaling* (`target_height=160`), dan peredaman *noise* dengan **Bilateral Filter** (`d=5, sigma=35`).
5. **Tahap 5 — Peningkatan Kontras Adaptif (*CLAHE Conditional*):** Menganalisis $\mu$ dan $\sigma$ histogram untuk menentukan apakah CLAHE perlu diterapkan.
6. **Tahap 6 — Penajaman Citra (*Unsharp Masking*):** Mempertegas tepi karakter dan menghitung skor ketajaman *Variance of Laplacian*.

In [ ]:
from Script.preprocessor import PlatePreprocessor

preprocessor = PlatePreprocessor()
prep_res = preprocessor.process(det["polarity_std"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].imshow(prep_res["smoothed"], cmap="gray")
axes[0].set_title(
    f"4. Grayscale & Bilateral Smoothing\n"
    f"Ukuran Upscale: {prep_res['smoothed'].shape[1]}x{prep_res['smoothed'].shape[0]} px",
    fontsize=10
)
axes[0].axis("off")

axes[1].imshow(prep_res["enhanced"], cmap="gray")
axes[1].set_title(
    f"5. Peningkatan Kontras (CLAHE Conditional)\n"
    f"Mean (μ)={prep_res['hist_mean']}, Std (σ)={prep_res['hist_std']}",
    fontsize=10
)
axes[1].axis("off")

axes[2].imshow(prep_res["sharpened"], cmap="gray")
axes[2].set_title(
    f"6. Penajaman Citra (Unsharp Masking)\n"
    f"Skor Ketajaman (Laplacian Var): {prep_res['sharpness_score']}",
    fontsize=10
)
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("=== EVALUASI BLOK 2 (TAHAP 4 - 6) ===")
print(f"• Statistik Histogram : Mean (μ) = {prep_res['hist_mean']} | Std (σ) = {prep_res['hist_std']}")
print(f"• Keputusan CLAHE     : {prep_res['clahe_status']}")
print(f"• Skor Unsharp Masking: Variance of Laplacian = {prep_res['sharpness_score']}")

---
## 5. Blok Segmentasi & Morfologi (Tahap 7 s/d 9): Biner, Morfologi & Pemisahan Dual-ROI (Utama vs Pajak)
Modul `PlateBinarizer` (`Script/binarizer.py`) menerima citra tajam berlatar gelap & bertuliskan terang, lalu menjalankan:
7. **Tahap 7 — Segmentasi Biner:** Thresholding (*Otsu* & *Adaptive*) menghasilkan latar hitam (`0`) dan karakter putih (`255`).
8. **Tahap 8 — Perbaikan Morfologi:** *Morphological Closing* pada karakter putih (`255`) untuk menyambung huruf/angka yang terputus.
9. **Tahap 9 — [Kebaruan 2 Bagian A] Segmentasi Dual-ROI:** Memisahkan citra menjadi **9a. ROI Nomor Utama (`roi_cleaned`)** dan **9b. ROI Nomor Pajak (`tax_roi_binary` & `tax_roi_gray`)** di bagian tengah-bawah pelat.

In [ ]:
from Script.binarizer import PlateBinarizer

binarizer = PlateBinarizer(method="otsu")
bin_res = binarizer.process(prep_res["sharpened"])

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

axes[0].imshow(bin_res["binary_raw"], cmap="gray")
axes[0].set_title(
    f"7. Segmentasi Biner (Otsu)\n"
    f"Rasio Piksel Putih: {bin_res['white_ratio']}%",
    fontsize=10
)
axes[0].axis("off")

axes[1].imshow(bin_res["morphed"], cmap="gray")
axes[1].set_title(
    "8. Perbaikan Morfologi\n(Morphological Closing)",
    fontsize=10
)
axes[1].axis("off")

axes[2].imshow(bin_res["roi_cleaned"], cmap="gray")
axes[2].set_title(
    f"9a. ROI Nomor Utama\n"
    f"Noise/Pajak Dibuang: {bin_res['removed_blobs']} blob",
    fontsize=10
)
axes[2].axis("off")

axes[3].imshow(bin_res["tax_roi_binary"], cmap="gray")
axes[3].set_title(
    "9b. ROI Nomor Pajak (MM.YY)\n(Sub-area Tengah-Bawah Pelat)",
    fontsize=10
)
axes[3].axis("off")

plt.tight_layout()
plt.show()

print("=== EVALUASI BLOK 3 (TAHAP 7 - 9) ===")
print(f"• Rasio Piksel Karakter Putih : {bin_res['white_ratio']}%")
print(f"• Komponen Non-Utama Dibuang  : {bin_res['removed_blobs']} komponen (bingkai, baut & angka pajak)")
print(f"• Ukuran ROI Nomor Utama      : {bin_res['roi_cleaned'].shape[1]}x{bin_res['roi_cleaned'].shape[0]} px")
print(f"• Ukuran ROI Nomor Pajak      : {bin_res['tax_roi_binary'].shape[1]}x{bin_res['tax_roi_binary'].shape[0]} px")

---
## 6. Blok Pengenalan Teks OCR, Validasi Regex & Klasifikasi Status Pajak (Tahap 10)
Modul `PlateRecognizer` (`Script/recognizer.py`) menjalankan:
* **Multi-Input OCR Voting:** Membaca 3 kandidat citra (*1. Biner ROI Utama Otsu*, *2. Grayscale Tajam Unsharp*, dan *3. Biner Alternatif Adaptive*) dan memilih pemenang berdasarkan validasi Regex + akumulasi *confidence*.
* **[Kebaruan 2 Bagian B] OCR Masa Berlaku Pajak (`recognize_tax`):** Membaca `MM.YY` pada ROI Pajak, mengoreksi huruf mirip angka (seperti `G1.28` $\rightarrow$ `01.28`), dan menentukan status **AKTIF** atau **MATI / KEDALUWARSA**.

In [ ]:
import torch
from Script.recognizer import PlateRecognizer
from Script.utils import get_ground_truth

use_gpu = torch.cuda.is_available()
recognizer = PlateRecognizer(use_gpu=use_gpu)

# 10A. Baca Nomor Pelat Utama (Komparasi 3 Kandidat Input)
ocr_res = recognizer.recognize(
    bin_res["roi_cleaned"],
    fallback_image=bin_res["upscaled_gray"],
    extra_images=bin_res["variants"][:1]
)

# 10B. Baca Nomor Pajak (MM.YY) & Status Aktif/Mati
tax_res = recognizer.recognize_tax(
    bin_res["tax_roi_binary"],
    tax_gray=bin_res["tax_roi_gray"]
)

gt_text = get_ground_truth(image_name, project_root)

print("=" * 72)
print("📊 TABEL KOMPARASI 3 KANDIDAT INPUT OCR (NOMOR UTAMA)")
print("=" * 72)
print(f"{'Kandidat Input Citra':<34} | {'Hasil Bacaan':<15} | {'Conf':<6} | {'Valid Regex'}")
print("-" * 72)
for cand in ocr_res["all_candidates"]:
    v_str = "YA" if cand["valid"] else "TIDAK"
    print(f"{cand['source']:<34} | {cand['fmt']:<15} | {cand['conf']:<6.2f} | {v_str}")
print("=" * 72)
print(f"🏆 PEMENANG NOMOR PLAT : {ocr_res['clean_text']} (Conf: {ocr_res['confidence']}, Sumber: {ocr_res['source']})")
if gt_text:
    match_str = "COCOK 100%" if ocr_res['clean_text'].replace(' ', '') == gt_text.replace(' ', '') else "BERBEDA"
    print(f"🎯 GROUND TRUTH        : {gt_text} ({match_str})")
print(f"🎨 WARNA & KENDARAAN   : Pelat {det['plate_color']} -> {det['vehicle_type']}")
print(f"📅 MASA BERLAKU PAJAK  : {tax_res['tax_text']} -> Status: {tax_res['tax_status']} (Conf: {tax_res['tax_confidence']})")
print("=" * 72)

---
## 7. Visualisasi Output Akhir & Ringkasan Box Evaluasi Lengkap (Tahap 11)
Menampilkan citra kendaraan dengan anotasi *Bounding Box + Leader Line + Badge 2 Baris (Nomor Pelat, Warna & Status Pajak)* beserta ringkasan visual seluruh tahapan.

In [ ]:
from Script.utils import draw_plate_annotations

det["text"] = ocr_res["clean_text"]
det["tax_text"] = tax_res["tax_text"]
det["tax_status"] = tax_res["tax_status"]
det["is_tax_active"] = tax_res["is_tax_active"]
det["is_occluded"] = (
    ocr_res["is_occluded"]
    or (not det["is_complete"])
    or (not ocr_res["is_valid_format"])
    or ocr_res["confidence"] < 0.4
)

annotated_img = draw_plate_annotations(image_bgr, [det])

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
plt.title(
    f"Output Akhir: {det['text']} | {det['vehicle_type']} | Pajak: {det['tax_text']} ({det['tax_status']})",
    fontsize=12
)
plt.axis("off")
plt.show()